# Μάθημα 03 - Πρότυπα Σχεδιασμού Πρακτόρων

Σε αυτό το μάθημα, εξερευνούμε τρία θεμελιώδη πρότυπα σχεδιασμού για την κατασκευή αποτελεσματικών πρακτόρων AI:

1. **Σαφείς Οδηγίες για τον Πράκτορα** — Δημιουργία ακριβών, ρόλου-οριστικών προτροπών που καθοδηγούν τη συμπεριφορά του πράκτορα
2. **Δομημένη Έξοδος με Μοντέλα Pydantic** — Εξασφάλιση ότι οι πράκτορες επιστρέφουν προβλέψιμα, ελεγμένα δεδομένα
3. **Πράκτορες με Μοναδική Ευθύνη** — Σχεδιασμός εστιασμένων πρακτόρων που κάνουν το καθένα ένα πράγμα καλά

Θα εφαρμόσουμε κάθε πρότυπο σε ένα σενάριο **προτάσεων προορισμού ταξιδιού**, χτίζοντας σταδιακά ένα σύστημα που μπορεί να προτείνει προορισμούς, να ελέγχει τη διαθεσιμότητα και να διαχειρίζεται τη λογιστική.


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Πρότυπο 1: Σαφείς Οδηγίες για τον Πράκτορα

Το πλέον αποτελεσματικό πρότυπο είναι επίσης το απλούστερο: η γραφή σαφών, λεπτομερειακών οδηγιών για τον πράκτορά σας.

Καλές οδηγίες ορίζουν:
- **Ποιος** είναι ο πράκτορας (προσωπικότητα και τόνος)
- **Τι** πρέπει να κάνει (ευθύνες βήμα προς βήμα)
- **Πώς** πρέπει να συμπεριφέρεται (περιορισμοί και στυλ)

Παρακάτω, δημιουργούμε έναν πράκτορα ταξιδιωτικού κονσιέρζ με ρητές οδηγίες που διαμορφώνουν κάθε απάντηση που παράγει.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Πρότυπο 2: Δομημένη έξοδος με μοντέλα Pydantic

Το κείμενο ελεύθερης μορφής είναι χρήσιμο για συνομιλία, αλλά τα υποκείμενα συστήματα χρειάζονται δομημένα δεδομένα.  
Συνδυάζοντας **μοντέλα Pydantic** με μια **λειτουργία εργαλείου**, μπορούμε:

- Να ορίσουμε ένα ακριβές σχήμα για την έξοδο του πράκτορα  
- Να επικυρώσουμε αυτόματα τις απαντήσεις  
- Να ενσωματώσουμε αξιόπιστα τα αποτελέσματα του πράκτορα στην λογική της εφαρμογής  

Εισάγουμε επίσης ένα εργαλείο που επιστρέφει λεπτομέρειες προορισμού ώστε ο πράκτορας να βασίζει τις προτάσεις του σε πραγματικά δεδομένα.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Agents με Μοναδική Ευθύνη

Οι πολύπλοκες εργασίες ωφελούνται από τη διαίρεση της εργασίας σε πολλούς εξειδικευμένους agents, καθένας με μία μοναδική ευθύνη:

- Έναν **Ειδικό Προορισμού** που γνωρίζει για μέρη και διαθεσιμότητα
- Έναν **Σχεδιαστή Logistics** που διαχειρίζεται πτήσεις, ξενοδοχεία και δρομολόγια

Αυτό αντικατοπτρίζει την αρχή του μηχανικού λογισμικού της *διαχωρισμού αρμοδιοτήτων* — κάθε agent είναι πιο εύκολο να δοκιμαστεί, να συντηρηθεί και να βελτιωθεί ανεξάρτητα.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Περίληψη

Σε αυτό το μάθημα εφαρμόσαμε τρία μοτίβα σχεδιασμού πρακτόρων σε ένα σενάριο συστήματος σύστασης ταξιδιών:

| Μοτίβο | Κύρια Ιδέα | Όφελος |
|---|---|---|
| **Σαφείς Οδηγίες** | Ορίστε τον χαρακτήρα, τις ευθύνες και τους περιορισμούς εκ των προτέρων | Συνεπής, σύμφωνα με το προφίλ του πράκτορα συμπεριφορά |
| **Δομημένο Αποτέλεσμα** | Χρησιμοποιήστε πρότυπα Pydantic ως μορφή απάντησης | Επικυρωμένα, από μηχανή αναγνώσιμα αποτελέσματα |
| **Ενιαία Ευθύνη** | Αναθέστε σε κάθε πράκτορα μία εστιασμένη εργασία | Ευκολότερη δοκιμή, συντήρηση και σύνθεση |

Αυτά τα μοτίβα συνδυάζονται φυσικά — μπορείτε να συνδυάσετε σαφείς οδηγίες με δομημένη έξοδο μέσα σε έναν πράκτορα με ενιαία ευθύνη για να δημιουργήσετε ανθεκτικά, έτοιμα για παραγωγή συστήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
